In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.output_parsers import PydanticOutputParser
from pydantic import BaseModel , Field
from typing import Literal
from langchain_core.runnables import RunnableParallel,RunnableBranch,RunnableLambda



c:\Users\KIIT0001\Desktop\Langchain-from-Scratch\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI (
    model="gemini-2.5-flash",
    temperature=0,
)

In [4]:
parser = StrOutputParser()

In [5]:
class Feedback(BaseModel):
    sentiment: Literal['positive','Negative'] = Field(description='Give the sentiment of the feedback')

In [6]:
parser2 = PydanticOutputParser(pydantic_object = Feedback)

In [7]:
prompt1 = PromptTemplate(
    template = 'Classify the sentiment of the following feedback text into positive or negative \n {feedback} \n {format_instruction}',
    input_variable = ['feedback'],
    partial_variables = {'format_instruction': parser2.get_format_instructions()}

)

In [8]:
classifier_chain = prompt1 | model | parser2

In [9]:
# result = classifier_chain.invoke({'feedback' : 'This is a terrible smartphone'})

Here we can see that we are getting positive or clearly a single word but we do not have control over it ...
The LLM can also print "The sentiment is Positive/Negative"!!

We need to ensure that the output always remains as Negative/Positive basically a single word defining the sentiment 
So that it works smoothly when used as input to others.

In [10]:
# print(result) ## prints Negative or positive in a json format

In [11]:
prompt2 = PromptTemplate(
    template = 'Write an appropriate response to this positive feedback \n {feedback}',
    input_variables= ['feedback']
)

In [12]:
prompt3 = PromptTemplate(
    template = 'Write an appropriate response to this negative feedback \n {feedback}',
    input_variables= ['feedback']
)

In [13]:
branch_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 | model | parser),
    (lambda x:x.sentiment == 'negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

In [14]:
chain = classifier_chain | branch_chain

In [17]:
print(chain.invoke({'feedback': 'This is a beautiful phone'}))

That's great news! Responding to positive feedback is a fantastic opportunity to reinforce customer loyalty and show appreciation.

Here are a few options, ranging from general to slightly more specific, depending on the context:

**1. General & Appreciative:**

*   "Thank you so much for your kind words! We truly appreciate your positive feedback."
*   "That's wonderful to hear! We're so glad you had a positive experience."
*   "We really appreciate you taking the time to share your feedback. It means a lot to us!"
*   "Thank you! We're delighted to know you're happy."

**2. Enthusiastic & Warm:**

*   "Wow, thank you! We're absolutely thrilled to hear that. Your feedback truly brightens our day!"
*   "That's fantastic! We're so happy we could provide you with a positive experience."
*   "We're over the moon to receive such positive feedback! Thank you for sharing."

**3. Slightly More Formal/Professional:**

*   "Thank you for your valuable feedback. We are pleased to learn that you 

In [16]:
chain.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
 +----------------------+  
 | PydanticOutputParser |  
 +----------------------+  
             *             
             *             
             *             
        +--------+         
        | Branch |         
        +--------+         
             *             
             *             
             *             
     +--------------+      
     | BranchOutput |      
     +--------------+      
